In [ ]:
import duckdb
import sys
from pathlib import Path
import pandas as pd 

ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
    
    
from src.db.sql_runner import run_query



# Antes de começar a Inteligência de Negócio, validar o dataset e realizar limpeza de acordo com as regras de negócio

In [12]:
df_nulos = run_query("01_nulls_check.sql")
df_nulos

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,pct_nulos,valores_unicos
0,id_24,DOUBLE,99.20,12
1,id_07,DOUBLE,99.13,99
2,id_08,DOUBLE,99.13,94
3,id_21,DOUBLE,99.13,469
4,id_25,DOUBLE,99.13,391
...,...,...,...,...
429,V317,DOUBLE,0.00,13197
430,V318,DOUBLE,0.00,10729
431,V319,DOUBLE,0.00,5914
432,V320,DOUBLE,0.00,7477


In [13]:
colunas_descarte = df_nulos[df_nulos['pct_nulos'] > 85]['column_name'].tolist()

colunas_mantidas = df_nulos[df_nulos['pct_nulos'] <= 85]['column_name'].tolist()

print(f'Total de {len(df_nulos)} colunas originais')
print(f'Total de {len(colunas_descarte)} colunas para descarte (>85% nulos)')
print(f'Total de {len(colunas_mantidas)} colunas mantidas (<=85% nulos)')

Total de 434 colunas originais
Total de 74 colunas para descarte (>85% nulos)
Total de 360 colunas mantidas (<=85% nulos)


In [14]:
df_emails = run_query("02_ordenar_emails.sql")
df_emails

,P_emaildomain,qtde_emails
0,gmail.com,228355
1,yahoo.com,100934
2,NaN,94456
3,hotmail.com,45250
4,anonymous.com,36998
5,aol.com,28289
6,comcast.net,7888
7,icloud.com,6267
8,outlook.com,5096
9,msn.com,4092


In [15]:
agrupar_emails_comprador = run_query('03_agrupar_emails_comprador.sql')
agrupar_emails_comprador

,qtde_por_provedor,categoria_provedor
0,228851,google
1,36998,anonymous
2,56564,outro
3,94456,missing
4,8225,apple
5,59477,microsoft
6,105969,yahoo


In [16]:
agrupar_emails_destino = run_query('04_agrupar_emails_destino.sql')
agrupar_emails_destino

,qtde_por_provedor,categoria_provedor
0,57242,google
1,20529,anonymous
2,453249,missing
3,9777,outro
4,2172,apple
5,33604,microsoft
6,13967,yahoo


In [17]:
colunas_descarte = df_nulos[df_nulos['pct_nulos'] > 85]['column_name'].tolist()
colunas_descarte_str = ', '.join(colunas_descarte)
colunas_descarte_str

'id_24, id_07, id_08, id_21, id_25, id_26, id_22, id_23, id_27, dist2, D7, id_18, D13, D14, D12, id_03, id_04, D6, id_33, D8, D9, id_09, id_10, id_30, id_32, id_34, id_14, V138, V152, V158, V159, V160, V161, V162, V163, V164, V165, V166, V153, V157, V155, V148, V149, V139, V140, V150, V151, V154, V147, V141, V142, V143, V144, V145, V146, V156, V338, V339, V337, V322, V323, V324, V325, V336, V327, V328, V329, V330, V331, V332, V333, V334, V335, V326'

In [18]:
view_limpa = run_query("05_view_limpa.sql", colunas_descarte_str=colunas_descarte_str)
view_limpa

,Count


In [19]:
query_limpa = run_query('06_test_view.sql')
query_limpa

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_29,id_31,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,categoria_provedor_comprador,categoria_provedor_destino
0,3008874,0,577096,150.0,H,10616,583.0,150.0,visa,226.0,...,NotFound,ie 11.0 for desktop,True,True,True,True,desktop,Trident/7.0,outro,missing
1,3008877,0,577152,15.0,H,6019,583.0,150.0,visa,226.0,...,Found,chrome 62.0,True,False,True,True,desktop,Windows,google,google
2,3008879,0,577158,50.0,H,11752,399.0,150.0,american express,119.0,...,NotFound,ie 11.0 for desktop,True,True,True,True,desktop,Trident/7.0,google,missing
3,3008884,0,577239,30.0,H,1680,555.0,150.0,visa,226.0,...,Found,firefox 57.0,True,False,True,True,desktop,Windows,outro,missing
4,3008887,0,577291,25.0,H,16075,514.0,150.0,mastercard,102.0,...,Found,chrome 62.0,True,False,True,True,desktop,MacOS,microsoft,missing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3574646,0,15722356,77.0,W,11815,206.0,150.0,mastercard,126.0,...,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,yahoo,missing
590536,3574647,0,15722378,117.0,W,7835,361.0,150.0,visa,226.0,...,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,outro,missing
590537,3574648,0,15722390,49.0,W,17307,225.0,150.0,mastercard,224.0,...,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,missing,missing
590538,3574649,0,15722429,77.0,W,12501,490.0,150.0,visa,226.0,...,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,yahoo,missing


In [ ]:
duckdb.execute(r"COPY vw_train_clean TO 'caminho para o arquivo' (FORMAT PARQUET)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Documentação: Tratamento de Dados e Engenharia de Features (Camada Silver)
Objetivo: Registrar o pipeline de limpeza, governança de nulos e normalização de categorias aplicados no dataset de detecção de fraudes para a consolidação da camada Silver (train_clean.parquet).
# 1. Regras de Governança e Qualidade de Dados (DQ)
Regra: Identificação e descarte de colunas que apresentem taxa de valores ausentes (nulos) superior a 85%.
Motivação: Variáveis com densidade de dados inferior a 15% introduzem ruído ao modelo e reduzem a eficiência computacional, sem agregar sinal preditivo relevante.
Resultado: 74 colunas excederam o limite do SLA e foram removidas dinamicamente do dataset através da cláusula EXCLUDE do DuckDB.
# 2. Engenharia de Features e Categorização (ID)
### Normalização de Domínios de E-mail (ID-02)
As colunas originais de e-mail do comprador (P_emaildomain) e do destinatário (R_emaildomain) apresentavam alta cardinalidade e fragmentação (ex: variações como gmail.com, gmail, hotmail.com, hotmail.co.uk).
Regra Aplicada: Mapeamento via sintaxe condicional CASE WHEN agrupando os domínios em 7 categorias estratégicas:

google

microsoft

yahoo

apple

anonymous

missing (preservação explícita de registros nulos para análise de risco)

outro (provedores corporativos/raros)
# Novas Features Geradas:
categoria_provedor_comprador: Provedor tratado do e-mail de quem realiza a compra.

categoria_provedor_destino: Provedor tratado do e-mail do destinatário (sinal valioso para fraudes em gift cards e entregas a terceiros).
# 4. Pipeline de Materialização e Exportação
Construção da View Virtual: As regras de negócio foram unificadas em uma VIEW lógica no DuckDB (vw_train_clean), evitando duplicação de dados na memória RAM durante os testes.

Exportação de Alta Performance: O arquivo final foi persistido diretamente do motor C++ do DuckDB para o disco, eliminando gargalos do PyArrow e otimizando o tempo de gravação.
Caminho de Destino: data/processed/train_clean.parquet
Tempo de Execução: 10,8 segundos para gravação de 590.540 linhas x 362 colunas.

# Pulando de camada (Bronze para Silver)
PARQUET_PATH do sql_runner alterado para parquet_limpo

In [36]:
risco_provedor_comprador = run_query('07_calculo_fraude_email_comp.sql')
risco_provedor_comprador

,categoria_provedor_comprador,total_transacoes,total_fraudes,porcentagem_fraude
0,microsoft,59477,3170.0,5.33
1,google,228851,9954.0,4.35
2,missing,94456,2790.0,2.95
3,apple,8225,238.0,2.89
4,anonymous,36998,859.0,2.32
5,outro,56564,1280.0,2.26
6,yahoo,105969,2372.0,2.24
